In [ ]:
# ==============================================================================
# MobileNetV2 U-Net for Masonry Crack Segmentation
# ==============================================================================
#
# Author : Farzaneh Zareian
#
# Description
# -----------
# This notebook implements a MobileNetV2-based U-Net architecture for binary
# semantic segmentation of masonry cracks.
#
# The complete workflow includes:
#   1. Dataset acquisition and preprocessing
#   2. Data augmentation using Albumentations
#   3. Hyperparameter optimization
#   4. Transfer learning with a pretrained MobileNetV2 encoder
#   5. Model training and evaluation using segmentation metrics
#   6. TensorFlow SavedModel export
#   7. ONNX conversion and inference validation
#
# ==============================================================================

In [ ]:
# ==============================================================================
# Install Required Packages
# ==============================================================================
!pip install -q GitPython
!pip install -q albumentations
!pip install -q tf2onnx==1.16.1
!pip install -q onnxruntime==1.17.3

In [ ]:
# ==============================================================================
# Import Libraries
# ==============================================================================

# Standard library
import itertools
import os
import random
from datetime import datetime

# Third-party libraries
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import tensorflow as tf
import tf2onnx
from git import Repo
from sklearn.model_selection import train_test_split

# TensorFlow / Keras
from tensorflow.keras import Model, backend as K, layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import TFSMLayer
from tensorflow.keras.optimizers import Adam, RMSprop

In [ ]:
# ==============================================================================
# Reproducibility
# ==============================================================================
# Set random seeds to improve experiment reproducibility.
# ==============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [ ]:
# ==============================================================================
# Dataset Acquisition
# ==============================================================================
# Clone the masonry crack segmentation dataset if it is not already available
# locally.
# ==============================================================================

DATASET_REPOSITORY = (
    "https://github.com/dimitrisdais/"
    "crack_detection_CNN_masonry.git"
)

DATASET_DIRECTORY = "crack_detection_CNN_masonry"

if not os.path.exists(DATASET_DIRECTORY):
    Repo.clone_from(DATASET_REPOSITORY, DATASET_DIRECTORY)

In [ ]:
# ==============================================================================
# Dataset Configuration
# ==============================================================================

IMAGE_DIR = os.path.join(
    DATASET_DIRECTORY,
    "dataset",
    "crack_detection_224_images",
)

MASK_DIR = os.path.join(
    DATASET_DIRECTORY,
    "dataset",
    "crack_detection_224_masks",
)

IMAGE_SIZE = (224, 224)

In [ ]:
# ==============================================================================
# Dataset Loading
# ==============================================================================

def load_data(
    image_dir: str,
    mask_dir: str,
    image_size: tuple = IMAGE_SIZE,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load RGB images and their corresponding binary segmentation masks.

    Parameters
    ----------
    image_dir : str
        Directory containing input images.

    mask_dir : str
        Directory containing ground-truth segmentation masks.

    image_size : tuple, optional
        Target image size as (height, width).

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Arrays containing normalized images and binary masks.
    """

    image_files = sorted(
        file
        for file in os.listdir(image_dir)
        if file.lower().endswith((".jpg", ".png"))
    )

    mask_files = sorted(
        file
        for file in os.listdir(mask_dir)
        if file.lower().endswith((".jpg", ".png"))
    )

    if len(image_files) != len(mask_files):
        raise ValueError("The number of images and masks does not match.")

    images = []
    masks = []

    for image_file, mask_file in zip(image_files, mask_files):

        image_path = os.path.join(image_dir, image_file)
        mask_path = os.path.join(mask_dir, mask_file)

        image = cv2.imread(image_path)

        if image is None:
            raise ValueError(f"Unable to read image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, image_size)
        image = image.astype(np.float32) / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None:
            raise ValueError(f"Unable to read mask: {mask_path}")

        mask = cv2.resize(mask, image_size)
        mask = (mask > 127).astype(np.float32)
        mask = mask[..., np.newaxis]

        images.append(image)
        masks.append(mask)

    return np.asarray(images), np.asarray(masks)

In [ ]:
# ==============================================================================
# Load Dataset
# ==============================================================================

X, y = load_data(IMAGE_DIR, MASK_DIR)

In [ ]:
# ==============================================================================
# Dataset Partitioning
# ==============================================================================
# Split the dataset into:
#   • Training   : 60%
#   • Validation : 20%
#   • Testing    : 20%
# ==============================================================================

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

In [ ]:
# ==============================================================================
# Dataset Summary
# ==============================================================================

print("Dataset loaded successfully.\n")
print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Testing samples    : {len(X_test)}")

In [ ]:
# ==============================================================================
# Data Augmentation Pipeline
# ==============================================================================

def get_training_augmentation() -> A.Compose:
    """
    Create the Albumentations augmentation pipeline.

    Returns
    -------
    albumentations.Compose
        Augmentation pipeline applied during training.
    """

    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.10,
                scale_limit=0.20,
                rotate_limit=30,
                p=0.50,
            ),
            A.RandomBrightnessContrast(p=0.50),
            A.ElasticTransform(p=0.10),
        ],
        additional_targets={"mask": "mask"},
    )

In [ ]:
# ==============================================================================
# Albumentations Data Generator
# ==============================================================================
# Applies online augmentation to image-mask pairs during training.
# ==============================================================================

def albumentations_generator(
    images: np.ndarray,
    masks: np.ndarray,
    batch_size: int = 4,
):
    """
    Generate augmented mini-batches indefinitely.

    Parameters
    ----------
    images : np.ndarray
        Training images.

    masks : np.ndarray
        Corresponding binary segmentation masks.

    batch_size : int, optional
        Number of samples per batch.

    Yields
    ------
    tuple[np.ndarray, np.ndarray]
        Augmented image and mask batches.
    """

    augmentation = get_training_augmentation()
    num_samples = len(images)

    while True:

        indices = np.random.permutation(num_samples)

        for start in range(0, num_samples, batch_size):

            batch_indices = indices[start : start + batch_size]

            if len(batch_indices) < batch_size:
                batch_indices = np.concatenate(
                    [
                        batch_indices,
                        np.random.choice(
                            num_samples,
                            batch_size - len(batch_indices),
                        ),
                    ]
                )

            batch_images = []
            batch_masks = []

            for idx in batch_indices:

                image = images[idx]
                mask = masks[idx]

                augmented = augmentation(
                    image=(image * 255).astype(np.uint8),
                    mask=(mask.squeeze() * 255).astype(np.uint8),
                )

                batch_images.append(
                    augmented["image"].astype(np.float32) / 255.0
                )

                batch_masks.append(
                    (augmented["mask"] > 127).astype(np.float32)[..., np.newaxis]
                )

            yield (
                np.asarray(batch_images),
                np.asarray(batch_masks),
            )

In [ ]:
# ==============================================================================
# TensorFlow Dataset Pipeline
# ==============================================================================
# Create TensorFlow datasets for model training and evaluation. During training,
# Albumentations is applied online, while validation and test datasets are
# batched without augmentation.
# ==============================================================================

def create_dataset(
    images: np.ndarray,
    masks: np.ndarray,
    batch_size: int,
    training: bool = False,
) -> tf.data.Dataset:
    """
    Create a TensorFlow dataset.

    Parameters
    ----------
    images : np.ndarray
        Input RGB images.

    masks : np.ndarray
        Corresponding binary segmentation masks.

    batch_size : int
        Number of samples per mini-batch.

    training : bool, optional
        Apply online data augmentation if True.

    Returns
    -------
    tf.data.Dataset
        Prepared TensorFlow dataset.
    """

    if training:

        dataset = tf.data.Dataset.from_generator(
            lambda: albumentations_generator(images, masks, batch_size),
            output_signature=(
                tf.TensorSpec(
                    shape=(None, *IMAGE_SIZE, 3),
                    dtype=tf.float32,
                ),
                tf.TensorSpec(
                    shape=(None, *IMAGE_SIZE, 1),
                    dtype=tf.float32,
                ),
            ),
        )

    else:

        dataset = tf.data.Dataset.from_tensor_slices((images, masks))
        dataset = dataset.batch(batch_size)

    return dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
# ==============================================================================
# Augmentation Visualization
# ==============================================================================
# Display randomly augmented image-mask pairs to verify the augmentation
# pipeline before model training.
# ==============================================================================

def visualize_augmented_samples(
    generator,
    num_samples: int = 5,
) -> None:
    """
    Display augmented image-mask pairs.

    Parameters
    ----------
    generator
        Albumentations data generator.

    num_samples : int, optional
        Number of samples to display.
    """

    images, masks = next(generator)
    num_samples = min(num_samples, len(images))

    fig, axes = plt.subplots(
        2,
        num_samples,
        figsize=(2.5 * num_samples, 6),
    )

    for i in range(num_samples):

        axes[0, i].imshow(images[i])
        axes[0, i].set_title(f"Sample {i + 1}")
        axes[0, i].axis("off")

        axes[1, i].imshow(
            masks[i].squeeze(),
            cmap="gray",
        )
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# Preview Training Augmentation
# ==============================================================================

preview_generator = albumentations_generator(
    X_train,
    y_train,
    batch_size=5,
)

visualize_augmented_samples(preview_generator)

In [ ]:
# ==============================================================================
# Loss Functions
# ==============================================================================
# Candidate loss functions evaluated during hyperparameter optimization.
# ==============================================================================

def f1_score_loss(epsilon: float = 1e-7):
    """
    F1-score loss.

    Minimizes (1 − F1 score), encouraging a balance between precision and
    recall.
    """

    def loss(y_true, y_pred):

        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        true_positive = tf.reduce_sum(y_true * y_pred)
        false_positive = tf.reduce_sum((1 - y_true) * y_pred)
        false_negative = tf.reduce_sum(y_true * (1 - y_pred))

        precision = true_positive / (
            true_positive + false_positive + epsilon
        )

        recall = true_positive / (
            true_positive + false_negative + epsilon
        )

        f1 = (
            2.0 * precision * recall
            / (precision + recall + epsilon)
        )

        return 1.0 - f1

    return loss


def focal_loss(
    gamma: float = 2.0,
    alpha: float = 0.25,
):
    """
    Focal loss.

    Down-weights well-classified pixels while emphasizing difficult examples.
    """

    def loss(y_true, y_pred):

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        cross_entropy = (
            -y_true * tf.math.log(y_pred)
            - (1 - y_true) * tf.math.log(1 - y_pred)
        )

        weights = (
            alpha * tf.pow(1 - y_pred, gamma) * y_true
            + (1 - alpha)
            * tf.pow(y_pred, gamma)
            * (1 - y_true)
        )

        return tf.reduce_mean(weights * cross_entropy)

    return loss


def weighted_cross_entropy(beta: float):
    """
    Weighted binary cross-entropy.

    Assigns additional weight to positive pixels to mitigate class imbalance.
    """

    def loss(y_true, y_pred):

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        return -tf.reduce_mean(
            beta * y_true * tf.math.log(y_pred)
            + (1 - y_true) * tf.math.log(1 - y_pred)
        )

    return loss

In [ ]:
# ==============================================================================
# Intersection over Union (IoU)
# ==============================================================================
# Primary evaluation metric for binary semantic segmentation.
# ==============================================================================

def iou_metric(
    y_true,
    y_pred,
    smooth: float = 1e-7,
):
    """
    Compute the Intersection over Union (IoU).
    """

    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)

    intersection = tf.reduce_sum(y_true * y_pred)

    union = (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        - intersection
    )

    return (intersection + smooth) / (union + smooth)

In [ ]:
# ==============================================================================
# MobileNetV2 U-Net Architecture
# ==============================================================================
# A pretrained MobileNetV2 encoder is combined with a U-Net-style decoder for
# binary semantic segmentation.
# ==============================================================================

def build_unet_with_mobilenet(
    input_shape: tuple = (*IMAGE_SIZE, 3),
    activation="relu",
) -> tf.keras.Model:
    """
    Build a MobileNetV2-based U-Net segmentation model.

    Parameters
    ----------
    input_shape : tuple, optional
        Input image dimensions.

    activation : str or callable, optional
        Decoder activation function.

    Returns
    -------
    tf.keras.Model
        MobileNetV2 U-Net model.
    """

    # ------------------------------------------------------------------
    # Encoder
    # ------------------------------------------------------------------

    encoder = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
    )

    encoder.trainable = False

    skip_layer_names = [
        "block_1_expand_relu",
        "block_3_expand_relu",
        "block_6_expand_relu",
        "block_13_expand_relu",
    ]

    skip_connections = [
        encoder.get_layer(name).output
        for name in skip_layer_names
    ]

    x = encoder.get_layer("block_16_project_BN").output

    # ------------------------------------------------------------------
    # Decoder
    # ------------------------------------------------------------------

    for skip in reversed(skip_connections):

        x = layers.UpSampling2D(
            size=(2, 2),
            interpolation="bilinear",
        )(x)

        x = layers.Conv2D(
            128,
            kernel_size=1,
            padding="same",
        )(x)

        x = layers.Resizing(
            skip.shape[1],
            skip.shape[2],
        )(x)

        x = layers.Concatenate()([x, skip])

        x = layers.Conv2D(
            128,
            kernel_size=3,
            padding="same",
            activation=activation,
        )(x)

        x = layers.Conv2D(
            128,
            kernel_size=3,
            padding="same",
            activation=activation,
        )(x)

    # ------------------------------------------------------------------
    # Output Layer
    # ------------------------------------------------------------------

    x = layers.UpSampling2D(
        size=(2, 2),
        interpolation="bilinear",
    )(x)

    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation=activation,
    )(x)

    outputs = layers.Conv2D(
        1,
        kernel_size=1,
        activation="sigmoid",
    )(x)

    return models.Model(
        inputs=encoder.input,
        outputs=outputs,
        name="MobileNetV2_UNet",
    )

In [ ]:
# ==============================================================================
# Hyperparameter Search
# ==============================================================================
# Perform a random search over multiple training configurations to identify the
# best-performing MobileNetV2 U-Net model based on validation IoU.
# ==============================================================================

# Candidate loss functions
LOSS_FUNCTIONS = {
    "binary_crossentropy": lambda: tf.keras.losses.BinaryCrossentropy(),
    "f1_score_loss": lambda: f1_score_loss(),
    "focal_loss": lambda: focal_loss(),
    "weighted_cross_entropy": lambda: weighted_cross_entropy(beta=10),
}

# Candidate decoder activation functions
ACTIVATIONS = [
    "relu",
    "elu",
    "swish",
]

# Candidate optimizers
OPTIMIZERS = {
    "adam": Adam,
    "rmsprop": RMSprop,
}

# Candidate learning rates
LEARNING_RATES = [
    5e-4,
    1e-3,
]

# Candidate batch sizes
BATCH_SIZES = [
    4,
    8,
]

In [ ]:
# ==============================================================================
# Generate Hyperparameter Configurations
# ==============================================================================

parameter_space = list(
    itertools.product(
        LOSS_FUNCTIONS.keys(),
        ACTIVATIONS,
        OPTIMIZERS.items(),
        LEARNING_RATES,
        BATCH_SIZES,
    )
)

SEARCH_SIZE = 25

sampled_configurations = random.sample(
    parameter_space,
    SEARCH_SIZE,
)

In [ ]:
# ==============================================================================
# Hyperparameter Optimization
# ==============================================================================

results = []
best_params = None
best_score = -np.inf

for (
    loss_name,
    activation_name,
    (optimizer_name, optimizer_class),
    learning_rate,
    batch_size,
) in sampled_configurations:

    print("\nTraining Configuration")
    print("-" * 60)

    print(
        f"Loss       : {loss_name}\n"
        f"Activation : {activation_name}\n"
        f"Optimizer  : {optimizer_name}\n"
        f"Learning Rate : {learning_rate}\n"
        f"Batch Size : {batch_size}"
    )

    # ------------------------------------------------------------------
    # Reset TensorFlow state
    # ------------------------------------------------------------------

    K.clear_session()
    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    # ------------------------------------------------------------------
    # Dataset preparation
    # ------------------------------------------------------------------

    train_dataset = create_dataset(
        X_train,
        y_train,
        batch_size=batch_size,
        training=True,
    )

    val_dataset = create_dataset(
        X_val,
        y_val,
        batch_size=batch_size,
        training=False,
    )

    steps_per_epoch = int(np.ceil(len(X_train) / batch_size))
    validation_steps = int(np.ceil(len(X_val) / batch_size))

    # ------------------------------------------------------------------
    # Build model
    # ------------------------------------------------------------------

    activation = (
        tf.keras.activations.swish
        if activation_name == "swish"
        else activation_name
    )

    model = build_unet_with_mobilenet(
        activation=activation
    )

    # ------------------------------------------------------------------
    # Compile model
    # ------------------------------------------------------------------

    optimizer = optimizer_class(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss=LOSS_FUNCTIONS[loss_name](),
        metrics=[
            "accuracy",
            iou_metric,
        ],
    )

    # ------------------------------------------------------------------
    # Train model
    # ------------------------------------------------------------------

    callbacks = [
        EarlyStopping(
            monitor="val_iou_metric",
            patience=5,
            mode="max",
            restore_best_weights=True,
            verbose=1,
        )
    ]

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=20,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        callbacks=callbacks,
        verbose=0,
    )

    # ------------------------------------------------------------------
    # Record experiment
    # ------------------------------------------------------------------

    best_epoch = np.argmax(history.history["val_iou_metric"])

    experiment = {
        "Loss": loss_name,
        "Activation": activation_name,
        "Optimizer": optimizer_name,
        "LR": learning_rate,
        "Batch_Size": batch_size,
        "Val_Loss": history.history["val_loss"][best_epoch],
        "Val_Accuracy": history.history["val_accuracy"][best_epoch],
        "Val_IoU": history.history["val_iou_metric"][best_epoch],
        "Epochs": len(history.history["loss"]),
    }

    results.append(experiment)

    if experiment["Val_IoU"] > best_score:
        best_score = experiment["Val_IoU"]
        best_params = experiment

In [ ]:
# ==============================================================================
# Best Hyperparameter Configuration
# ==============================================================================

print("\nBest Hyperparameter Configuration")
print("=" * 50)
print(f"Best Validation IoU: {best_score:.4f}\n")

for key, value in best_params.items():
    print(f"{key:<12}: {value}")

In [ ]:
# ==============================================================================
# Final Model Training
# ==============================================================================
# Rebuild and train the MobileNetV2 U-Net model using the best-performing
# hyperparameter configuration identified during random search.
# ==============================================================================

In [ ]:
# ==============================================================================
# Build Model
# ==============================================================================

activation = (
    tf.keras.activations.swish
    if best_params["Activation"] == "swish"
    else best_params["Activation"]
)

model = build_unet_with_mobilenet(
    activation=activation
)

In [ ]:
# ==============================================================================
# Compile Model
# ==============================================================================

optimizer = (
    Adam(learning_rate=best_params["LR"])
    if best_params["Optimizer"] == "adam"
    else RMSprop(learning_rate=best_params["LR"])
)

model.compile(
    optimizer=optimizer,
    loss=LOSS_FUNCTIONS[best_params["Loss"]](),
    metrics=[
        "accuracy",
        iou_metric,
    ],
)

In [ ]:
# ==============================================================================
# Dataset Preparation
# ==============================================================================

batch_size = best_params["Batch_Size"]

train_dataset = create_dataset(
    X_train,
    y_train,
    batch_size=batch_size,
    training=True,
)

val_dataset = create_dataset(
    X_val,
    y_val,
    batch_size=batch_size,
    training=False,
)

In [ ]:
# ==============================================================================
# Training Callbacks
# ==============================================================================
# Early stopping restores the model weights from the epoch with the highest
# validation IoU to reduce overfitting.
# ==============================================================================

callbacks = [
    EarlyStopping(
        monitor="val_iou_metric",
        patience=10,
        mode="max",
        restore_best_weights=True,
        verbose=1,
    )
]

In [ ]:
# ==============================================================================
# Model Training
# ==============================================================================

steps_per_epoch = int(np.ceil(len(X_train) / batch_size))
validation_steps = int(np.ceil(len(X_val) / batch_size))

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ==============================================================================
# Save Trained Model
# ==============================================================================
# Save the trained model in native Keras format for future inference,
# fine-tuning, or model conversion.
# ==============================================================================

MODEL_PATH = "mobilenetv2_unet.keras"

model.save(MODEL_PATH)

print(f"Model saved successfully: {MODEL_PATH}")

In [ ]:
# ==============================================================================
# Training History Extraction
# ==============================================================================

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

train_accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]

train_iou = history.history.get("iou_metric", [])
val_iou = history.history.get("val_iou_metric", [])

epochs = range(1, len(train_loss) + 1)

In [ ]:
# ==============================================================================
# Training Performance Visualization
# ==============================================================================
# Plot training and validation curves for:
#   • Loss
#   • Accuracy
#   • IoU
# ==============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4),
)

# Loss
axes[0].plot(
    epochs,
    train_loss,
    label="Training Loss",
)

axes[0].plot(
    epochs,
    val_loss,
    label="Validation Loss",
)

axes[0].set_title("Loss")
axes[0].legend()


# Accuracy
axes[1].plot(
    epochs,
    train_accuracy,
    label="Training Accuracy",
)

axes[1].plot(
    epochs,
    val_accuracy,
    label="Validation Accuracy",
)

axes[1].set_title("Accuracy")
axes[1].legend()


# IoU
axes[2].plot(
    epochs,
    train_iou,
    label="Training IoU",
)

axes[2].plot(
    epochs,
    val_iou,
    label="Validation IoU",
)

axes[2].set_title("IoU")
axes[2].legend()


plt.tight_layout()

plt.savefig(
    "training_metrics.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ==============================================================================
# Model Evaluation
# ==============================================================================
# Evaluate segmentation performance using:
#   • Accuracy
#   • Precision
#   • Recall
#   • F1-score
#   • Intersection over Union (IoU)
# ==============================================================================


def evaluate_model(
    model: tf.keras.Model,
    images: np.ndarray,
    masks: np.ndarray,
    dataset_name: str = "Validation",
    save_dir: str = "evaluation_results",
) -> None:
    """
    Evaluate segmentation performance and save prediction examples.

    Parameters
    ----------
    model : tf.keras.Model
        Trained segmentation model.

    images : np.ndarray
        Evaluation images.

    masks : np.ndarray
        Ground truth segmentation masks.

    dataset_name : str, optional
        Dataset name used in reports and filenames.

    save_dir : str, optional
        Directory for saving prediction visualizations.
    """

    os.makedirs(save_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Generate predictions
    # ------------------------------------------------------------------

    predictions = model.predict(
        images,
        verbose=1,
    )

    predicted_masks = (
        predictions > 0.5
    ).astype(np.float32)


    # ------------------------------------------------------------------
    # Crack tolerance evaluation using mask dilation
    # ------------------------------------------------------------------

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3),
    )

    dilated_masks = np.array(
        [
            cv2.dilate(
                mask.squeeze(),
                kernel,
                iterations=1,
            )
            for mask in masks
        ]
    )[..., np.newaxis]


    # ------------------------------------------------------------------
    # Confusion matrix calculation
    # ------------------------------------------------------------------

    true_positive = np.sum(
        np.logical_and(
            predicted_masks == 1,
            np.logical_or(
                masks == 1,
                dilated_masks == 1,
            ),
        )
    )

    false_positive = np.sum(
        np.logical_and(
            predicted_masks == 1,
            np.logical_and(
                masks == 0,
                dilated_masks == 0,
            ),
        )
    )

    false_negative = np.sum(
        np.logical_and(
            predicted_masks == 0,
            masks == 1,
        )
    )

    true_negative = np.sum(
        np.logical_and(
            predicted_masks == 0,
            masks == 0,
        )
    )


    # ------------------------------------------------------------------
    # Metrics
    # ------------------------------------------------------------------

    epsilon = 1e-7

    accuracy = (
        (true_positive + true_negative)
        /
        (
            true_positive
            + true_negative
            + false_positive
            + false_negative
            + epsilon
        )
    )

    precision = (
        true_positive
        /
        (true_positive + false_positive + epsilon)
    )

    recall = (
        true_positive
        /
        (true_positive + false_negative + epsilon)
    )

    f1_score = (
        2 * precision * recall
        /
        (precision + recall + epsilon)
    )

    intersection = np.logical_and(
        masks == 1,
        predicted_masks == 1,
    ).sum()

    union = np.logical_or(
        masks == 1,
        predicted_masks == 1,
    ).sum()

    iou = (
        intersection + epsilon
    ) / (
        union + epsilon
    )


    # ------------------------------------------------------------------
    # Display metrics
    # ------------------------------------------------------------------

    print(f"\n{dataset_name} Set Evaluation")
    print("=" * 40)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1_score:.4f}")
    print(f"IoU      : {iou:.4f}")


    # ------------------------------------------------------------------
    # Save prediction visualizations
    # ------------------------------------------------------------------

    sample_indices = np.random.choice(
        len(images),
        size=min(5, len(images)),
        replace=False,
    )

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


    for sample_id, index in enumerate(sample_indices):

        fig, axes = plt.subplots(
            1,
            4,
            figsize=(12, 3),
        )

        axes[0].imshow(images[index])
        axes[0].set_title("Input")
        axes[0].axis("off")

        axes[1].imshow(
            masks[index].squeeze(),
            cmap="gray",
        )
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")


        probability_map = axes[2].imshow(
            predictions[index].squeeze(),
            cmap="viridis",
            vmin=0,
            vmax=1,
        )

        axes[2].set_title("Prediction")
        axes[2].axis("off")

        fig.colorbar(
            probability_map,
            ax=axes[2],
        )


        axes[3].imshow(images[index])

        axes[3].imshow(
            predicted_masks[index].squeeze(),
            cmap="Reds",
            alpha=0.4,
        )

        axes[3].set_title("Overlay")
        axes[3].axis("off")


        plt.tight_layout()

        filename = (
            f"{dataset_name}_sample_{sample_id + 1}_"
            f"{timestamp}.png"
        )

        filepath = os.path.join(
            save_dir,
            filename,
        )

        plt.savefig(
            filepath,
            dpi=600,
            bbox_inches="tight",
        )

        plt.close(fig)

        print(f"Saved: {filepath}")

In [ ]:
# ==============================================================================
# Validation and Test Evaluation
# ==============================================================================

evaluate_model(
    model,
    X_val,
    y_val,
    dataset_name="Validation",
)

evaluate_model(
    model,
    X_test,
    y_test,
    dataset_name="Test",
)

In [ ]:
# ==============================================================================
# TensorFlow SavedModel Export
# ==============================================================================
#
# The trained MobileNetV2 U-Net model is exported as a TensorFlow SavedModel.
# This intermediate format is used as the source model for ONNX conversion.
#
# ==============================================================================

print("\nExporting TensorFlow SavedModel...")

# Switch model to inference mode before export
model.trainable = False


saved_model_dir = "segmentation_savedmodel"


model.export(saved_model_dir)


print(
    f"SavedModel exported successfully: {saved_model_dir}"
)

In [ ]:
# ==============================================================================
# Prepare Keras-Compatible Model for ONNX Conversion
# ==============================================================================
#
# TensorFlow SavedModel is loaded through TFSMLayer to create a Keras-compatible
# inference graph required by tf2onnx conversion.
#
# ==============================================================================

print("\nPreparing model for ONNX conversion...")


saved_model_layer = TFSMLayer(
    saved_model_dir,
    call_endpoint="serving_default",
)


onnx_input = tf.keras.Input(
    shape=(224, 224, 3),
    dtype=tf.float32,
    name="input",
)


onnx_output = saved_model_layer(
    onnx_input
)


# Extract output tensor when SavedModel returns a dictionary
if isinstance(onnx_output, dict):

    onnx_output = list(
        onnx_output.values()
    )[0]


onnx_model = Model(
    inputs=onnx_input,
    outputs=onnx_output,
)


print(
    "ONNX conversion model prepared successfully."
)

In [ ]:
# ==============================================================================
# ONNX Model Conversion
# ==============================================================================
#
# Convert the TensorFlow inference model into ONNX format using tf2onnx.
#
# ==============================================================================

print("\nConverting model to ONNX format...")


onnx_output_path = (
    "masonry_segmentation_v1.onnx"
)


conversion_signature = [
    tf.TensorSpec(
        shape=(1, 224, 224, 3),
        dtype=tf.float32,
        name="input",
    )
]


tf2onnx.convert.from_keras(
    onnx_model,
    input_signature=conversion_signature,
    opset=13,
    output_path=onnx_output_path,
)


print(
    "ONNX conversion completed successfully."
)

print(
    f"ONNX model saved: {onnx_output_path}"
)

In [ ]:
# ==============================================================================
# ONNX Runtime Inference Validation
# ==============================================================================
#
# Validate the exported ONNX model by:
#
#   1. Loading the model with ONNX Runtime
#   2. Creating a dummy input tensor
#   3. Running inference
#   4. Checking output dimensions
#
# ==============================================================================

print("\nValidating ONNX inference...")


onnx_session = ort.InferenceSession(
    onnx_output_path,
    providers=[
        "CPUExecutionProvider"
    ],
)


input_tensor_name = (
    onnx_session
    .get_inputs()[0]
    .name
)


output_tensor_name = (
    onnx_session
    .get_outputs()[0]
    .name
)


print(
    f"ONNX input tensor : {input_tensor_name}"
)

print(
    f"ONNX output tensor: {output_tensor_name}"
)


# Generate a random inference sample
dummy_input = np.random.rand(
    1,
    224,
    224,
    3,
).astype(
    np.float32
)


onnx_prediction = onnx_session.run(
    [output_tensor_name],
    {
        input_tensor_name: dummy_input
    },
)


prediction_shape = (
    onnx_prediction[0].shape
)


print(
    "\nONNX inference completed successfully."
)

print(
    f"Output prediction shape: {prediction_shape}"
)